**Sarcasm Preservation in Machine Translation: Effects on Automatic Detection in English–Russian Settings** - Master thesis

Part 1 - Sarcasm detetion (+ evaluation, visualization)

Part 2 - Machine Translation (EN-RU)

Part 3 - Sarcasm detection on the MT-text (+ evaluation, visualization)

Part 4 - Comparison of translation techniques

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

BASE_PATH = "/content/drive/MyDrive/sarcasm_thesis"

folders = [
    "models",
    "results",
    "logs",
    "plots",
    "data"
]

for folder in folders:
    os.makedirs(os.path.join(BASE_PATH, folder), exist_ok=True)

In [ ]:
!pip install transformers datasets scikit-learn matplotlib seaborn

In [ ]:
import torch
print(torch.cuda.is_available())

False


In [ ]:
from datasets import load_dataset

dataset_ru = load_dataset("Kostya165/ru_emotion_dvach")
df_ru = dataset_ru["train"].to_pandas()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

train_part_1.csv: 0.00B [00:00, ?B/s]

train_part_2.csv: 0.00B [00:00, ?B/s]

train_part_3.csv: 0.00B [00:00, ?B/s]

...

train_part_5.csv: 0.00B [00:00, ?B/s]

train_part_6.csv: 0.00B [00:00, ?B/s]

valid.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/59061 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2507 [00:00<?, ? examples/s]

In [ ]:
df_ru.head()
df_ru.columns

Index(['text', 'label'], dtype='object')

In [ ]:
import pandas as pd

df_en = pd.read_json(f"{BASE_PATH}/Sarcasm_Headlines_Dataset.json", lines=True)

In [ ]:
df_en = df_en.rename(columns={
    "headline": "text",
    "is_sarcastic": "label"
})

In [ ]:
df_en = df_en[["text", "label"]].dropna()
df_ru = df_ru[["text", "label"]].dropna()

In [ ]:
df_ru["label"] = df_ru["label"].apply(lambda x: 1 if x == "sarcasm" else 0)

In [ ]:
df_en.head()

,text,label
0,former versace store clerk sues over secret 'b...,0
1,the 'roseanne' revival catches up to our thorn...,0
2,mom starting to fear son's web series closest ...,1
3,"boehner just wants wife to listen, not come up...",1
4,j.k. rowling wishes snape happy birthday in th...,0


In [ ]:
df_en['label'].value_counts()

,count
label,
0,14985
1,11724


In [ ]:
df_ru.head()

,text,label
0,"Я боюсь, что из-за контроля сварных соединений...",0
1,"О, ДМС - это просто сказка для бедных! Кто бы ...",1
2,Это потрясающий фильм! Я в восторге! 10/10,0
3,"Ой, какие же законы и судебная практика разные...",1
4,"а что если они правы, что если войн не станет ...",0


In [ ]:
df_ru['label'].value_counts()

,count
label,
0,47361
1,11693


In [ ]:
df_sarcasm = df_ru[df_ru['label'] == 1]
df_non = df_ru[df_ru['label'] == 0]


df_non_sampled = df_non.sample(n=len(df_sarcasm)*2, random_state=42)

df_ru_balanced = pd.concat([df_sarcasm, df_non_sampled]).sample(frac=1, random_state=42)

In [ ]:
df_ru_balanced['label'].value_counts()

,count
label,
0,23386
1,11693


In [ ]:
from sklearn.model_selection import train_test_split

train_val_ru, test_ru = train_test_split(
    df_ru_balanced,
    test_size=0.15,
    stratify=df_ru_balanced['label'],
    random_state=42
)

train_ru, val_ru = train_test_split(
    train_val_ru,
    test_size=0.176,
    stratify=train_val_ru['label'],
    random_state=42
)

In [ ]:
train_ru.to_csv(f"{BASE_PATH}/data/train_ru.csv", index=False)
val_ru.to_csv(f"{BASE_PATH}/data/val_ru.csv", index=False)
test_ru.to_csv(f"{BASE_PATH}/data/test_ru.csv", index=False)

In [ ]:
train_val_en, test_en = train_test_split(
    df_en,
    test_size=0.15,
    stratify=df_en['label'],
    random_state=42
)

train_en, val_en = train_test_split(
    train_val_en,
    test_size=0.176,
    stratify=train_val_en['label'],
    random_state=42
)

In [ ]:
train_en.to_csv(f"{BASE_PATH}/data/train_en.csv", index=False)
val_en.to_csv(f"{BASE_PATH}/data/val_en.csv", index=False)
test_en.to_csv(f"{BASE_PATH}/data/test_en.csv", index=False)

# **Part 1**

**Model:** https://huggingface.co/FacebookAI/xlm-roberta-base

**Dataset RU:** https://huggingface.co/datasets/Kostya165/ru_emotion_dvach (sarcasm is separately marked)

**Dataset EN:** https://www.kaggle.com/datasets/rmisra/news-headlines-dataset-for-sarcasm-detection


**Training combinations for fine-tuning:**

1. Only EN

2. Only RU

3. Combined EN + RU

4. Sequential fine-tuning: First EN and then RU




Outcome - choose the best model to use in part 3. It is important that the model can sucsesfully detect sarcasm in both languages so that post-MT-text results would be valid

**Only EN**

In [ ]:
import pandas as pd

train_en = pd.read_csv(f"{BASE_PATH}/data/train_en.csv")
val_en = pd.read_csv(f"{BASE_PATH}/data/val_en.csv")
test_en = pd.read_csv(f"{BASE_PATH}/data/test_en.csv")

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("FacebookAI/xlm-roberta-base")

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True
    )

In [ ]:
from datasets import Dataset

train_en_ds = Dataset.from_pandas(train_en)
val_en_ds = Dataset.from_pandas(val_en)
test_en_ds = Dataset.from_pandas(test_en)

In [ ]:
train_en_ds = train_en_ds.map(tokenize, batched=True)
val_en_ds = val_en_ds.map(tokenize, batched=True)
test_en_ds = test_en_ds.map(tokenize, batched=True)

Map:   0%|          | 0/18706 [00:00<?, ? examples/s]

Map:   0%|          | 0/3996 [00:00<?, ? examples/s]

Map:   0%|          | 0/4007 [00:00<?, ? examples/s]

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "FacebookAI/xlm-roberta-base",
    num_labels=2
)

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: FacebookAI/xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir=f"{BASE_PATH}/models/en_only",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir=f"{BASE_PATH}/logs",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    learning_rate=2e-5,
    weight_decay=0.01
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "f1": f1}

In [ ]:
from transformers import Trainer, DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_en_ds,
    eval_dataset=val_en_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()
trainer.evaluate()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.298222,0.324049,0.894645,0.870898
2,0.217705,0.301782,0.913914,0.897983
3,0.168838,0.509081,0.889640,0.861712
4,0.109137,0.382239,0.921672,0.909302
5,0.076662,0.468631,0.917668,0.902112


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'eval_loss': 0.4686310291290283,
 'eval_accuracy': 0.9176676676676677,
 'eval_f1': 0.9021124665278191,
 'eval_runtime': 8.119,
 'eval_samples_per_second': 492.18,
 'eval_steps_per_second': 30.792,
 'epoch': 5.0}

In [ ]:
trainer.train()
trainer.evaluate()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.692646,0.686544,0.561061,0.000000
2,0.687821,0.673946,0.561061,0.000000


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


{'eval_loss': 0.6739462614059448,
 'eval_accuracy': 0.561061061061061,
 'eval_f1': 0.0,
 'eval_runtime': 6.935,
 'eval_samples_per_second': 576.211,
 'eval_steps_per_second': 36.049,
 'epoch': 2.0}

In [ ]:
trainer.save_model(f"{BASE_PATH}/models/en_only_final")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
results = trainer.evaluate()
print(results)

{'eval_loss': 0.4686310291290283, 'eval_accuracy': 0.9176676676676677, 'eval_f1': 0.9021124665278191, 'eval_runtime': 8.8874, 'eval_samples_per_second': 449.625, 'eval_steps_per_second': 28.13, 'epoch': 5.0}


**Only RU**

In [ ]:
import pandas as pd

train_ru = pd.read_csv(f"{BASE_PATH}/data/train_ru.csv")
val_ru = pd.read_csv(f"{BASE_PATH}/data/val_ru.csv")
test_ru = pd.read_csv(f"{BASE_PATH}/data/test_ru.csv")

In [ ]:
train_ru = train_ru.sample(n=18706, random_state=42)
val_ru = val_ru.sample(n=3996, random_state=42)
test_ru = test_ru.sample(n=4007, random_state=42)

In [ ]:
from datasets import Dataset

train_ru_ds = Dataset.from_pandas(train_ru)
val_ru_ds = Dataset.from_pandas(val_ru)
test_ru_ds = Dataset.from_pandas(test_ru)

In [ ]:
train_ru_ds = train_ru_ds.map(tokenize, batched=True)
val_ru_ds = val_ru_ds.map(tokenize, batched=True)
test_ru_ds = test_ru_ds.map(tokenize, batched=True)

Map:   0%|          | 0/24569 [00:00<?, ? examples/s]

Map:   0%|          | 0/5248 [00:00<?, ? examples/s]

Map:   0%|          | 0/5262 [00:00<?, ? examples/s]

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir=f"{BASE_PATH}/models/ru_only",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir=f"{BASE_PATH}/logs",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    learning_rate=2e-5,
    weight_decay=0.01
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "f1": f1}

In [ ]:
from transformers import Trainer, DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ru_ds,
    eval_dataset=val_ru_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()
trainer.evaluate()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.185576,0.211969,0.940358,0.914082
2,0.139004,0.205789,0.950267,0.921409
3,0.102447,0.261133,0.954840,0.929275
4,0.065654,0.242175,0.957317,0.935072
5,0.036330,0.276164,0.958270,0.936833


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'eval_loss': 0.2761636972427368,
 'eval_accuracy': 0.9582698170731707,
 'eval_f1': 0.9368329968272282,
 'eval_runtime': 41.0571,
 'eval_samples_per_second': 127.822,
 'eval_steps_per_second': 7.989,
 'epoch': 5.0}

In [ ]:
results = trainer.evaluate()
print(results)

{'eval_loss': 0.2761636972427368, 'eval_accuracy': 0.9582698170731707, 'eval_f1': 0.9368329968272282, 'eval_runtime': 40.2129, 'eval_samples_per_second': 130.506, 'eval_steps_per_second': 8.157, 'epoch': 5.0}


In [ ]:
trainer.save_model(f"{BASE_PATH}/models/ru_only_final")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

**Combined EN + RU**

In [ ]:
import pandas as pd

train_en = pd.read_csv(f"{BASE_PATH}/data/train_en.csv")
val_en = pd.read_csv(f"{BASE_PATH}/data/val_en.csv")
test_en = pd.read_csv(f"{BASE_PATH}/data/test_en.csv")

train_ru = pd.read_csv(f"{BASE_PATH}/data/train_ru.csv")
val_ru = pd.read_csv(f"{BASE_PATH}/data/val_ru.csv")
test_ru = pd.read_csv(f"{BASE_PATH}/data/test_ru.csv")

In [ ]:
train_en_small = train_en.sample(n=9353, random_state=42)
train_ru_small = train_ru.sample(n=9353, random_state=42)

val_en_small = val_en.sample(n=1998, random_state=42)
val_ru_small = val_ru.sample(n=1998, random_state=42)

test_en_small = test_en.sample(n=2003, random_state=42)
test_ru_small = test_ru.sample(n=2004, random_state=42)

In [ ]:
train_combined = pd.concat([train_en_small, train_ru_small]).sample(frac=1, random_state=42)
val_combined = pd.concat([val_en_small, val_ru_small]).sample(frac=1, random_state=42)
test_combined = pd.concat([test_en_small, test_ru_small]).sample(frac=1, random_state=42)

In [ ]:
train_combined.to_csv(
    f"{BASE_PATH}/data/train_combined.csv",
    index=False
)

val_combined.to_csv(
    f"{BASE_PATH}/data/val_combined.csv",
    index=False
)

test_combined.to_csv(
    f"{BASE_PATH}/data/test_combined.csv",
    index=False
)

In [ ]:
train_combined['label'].value_counts()

,count
label,
0,11460
1,7246


In [ ]:
from datasets import Dataset

train_combined_ds = Dataset.from_pandas(train_combined)
val_combined_ds = Dataset.from_pandas(val_combined)
test_combined_ds = Dataset.from_pandas(test_combined)

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("FacebookAI/xlm-roberta-base")

def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding=True
    )

In [ ]:
train_combined_ds = train_combined_ds.map(tokenize, batched=True)
val_combined_ds = val_combined_ds.map(tokenize, batched=True)
test_combined_ds = test_combined_ds.map(tokenize, batched=True)

Map:   0%|          | 0/18706 [00:00<?, ? examples/s]

Map:   0%|          | 0/3996 [00:00<?, ? examples/s]

Map:   0%|          | 0/4007 [00:00<?, ? examples/s]

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir=f"{BASE_PATH}/models/combined",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir=f"{BASE_PATH}/logs",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    learning_rate=2e-5,
    weight_decay=0.01
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "f1": f1}

In [ ]:
from transformers import Trainer, DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_combined_ds,
    eval_dataset=val_combined_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()
trainer.evaluate()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.264325,0.319381,0.888388,0.835788
2,0.185204,0.278130,0.918669,0.885200
3,0.121919,0.342869,0.928679,0.904137
4,0.079713,0.386064,0.930430,0.908852
5,0.045957,0.434209,0.933183,0.909828


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'eval_loss': 0.43420925736427307,
 'eval_accuracy': 0.9331831831831832,
 'eval_f1': 0.9098277608915907,
 'eval_runtime': 112.4465,
 'eval_samples_per_second': 35.537,
 'eval_steps_per_second': 2.223,
 'epoch': 5.0}

In [ ]:
results = trainer.evaluate()
print(results)

{'eval_loss': 0.43420925736427307, 'eval_accuracy': 0.9331831831831832, 'eval_f1': 0.9098277608915907, 'eval_runtime': 111.8396, 'eval_samples_per_second': 35.73, 'eval_steps_per_second': 2.235, 'epoch': 5.0}


In [ ]:
trainer.save_model(f"{BASE_PATH}/models/combined_only_final")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

# **Part 2**

Model: https://huggingface.co/docs/transformers/model_doc/nllb

Dataset EN: https://www.kaggle.com/datasets/rmisra/news-headlines-dataset-for-sarcasm-detection

In [ ]:
import pandas as pd

test_en = pd.read_csv(f"{BASE_PATH}/data/test_en.csv")
test_en.head()

,text,label
0,study: you have hpv,1
1,the justice department pledge to prosecute whi...,0
2,every new yorker found murdered,1
3,nostalgic man can still remember time when bil...,1
4,'steven universe' is exploring unhealthy relat...,0


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "facebook/nllb-200-distilled-600M"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


model.safetensors:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

In [ ]:
def translate(texts):
    tokenizer.src_lang = "eng_Latn"

    inputs = tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=128
    )

    translated_tokens = model.generate(
        **inputs,
        forced_bos_token_id=tokenizer.convert_tokens_to_ids("rus_Cyrl"),
        max_length=128
    )

    return tokenizer.batch_decode(translated_tokens, skip_special_tokens=True)

In [ ]:
translated_texts = []

for i in range(0, len(test_en), batch_size):
    batch = test_en["text"][i:i+batch_size].tolist()
    translated = translate(batch)
    translated_texts.extend(translated)

KeyboardInterrupt: 

In [ ]:
test_en["translated_text"] = translated_texts

In [ ]:
test_en.to_csv(f"{BASE_PATH}/data/test_en_translated_ru.csv", index=False)

# **Part 3**

**Model (the best from Part 1):**

**Dataset (MT EN->RU):**